# Temporal Feature Analysis — Weekend Gap Prediction

**Dataset:** `dataset_clean_temporal.csv` — 8,437 Monday rows, 25 tickers, April 2016 – December 2024.  
**Goal:** Evaluate whether cyclic temporal encodings (`Month_sin/cos`, `WeekOfYear_sin/cos`, `Quarter_sin/cos`, `DaysSinceStart`) lift AUC above the project's 15-feature baseline.

### Notebook structure

| Section | What |
|---------|------|
| 1. Setup & Load | Imports, data load, primary set filtering |
| 2. EDA Plots | Display pre-generated temporal EDA figures |
| 3. Temporal Feature Stats | Correlation with `GapUp`, seasonal GapUp rates |
| 4. Feature Sets | Baseline 15, Temporal-only, Combined |
| 5. Walk-forward Evaluation | LogReg + XGBoost across 6 folds (2019-2024) |
| 6. Results Comparison | AUC lift table + bar chart |
| 7. Summary | Interpretation and takeaways |

### Leakage discipline
- COVID rows (`is_extreme_event == 1`, Feb-May 2020) excluded from all training and evaluation.
- Leaky weekly price aggregates (`WeeklyReturn`, `IntraWeekVolatility`, `FridayPosition`, `OpenCloseSpread`) never referenced.
- All temporal features are computed from the calendar date only — no future information.

## 1. Setup & Load

### 1a. Imports

Standard data science stack. We import `pointbiserialr` from `scipy.stats` to measure each temporal feature's linear correlation with the binary `GapUp` target. `xgboost` is imported inside a try/except so the notebook degrades gracefully if the package is absent — all XGBoost cells are gated behind the `_HAS_XGB` flag.

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from scipy.stats import pointbiserialr
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score

try:
    from xgboost import XGBClassifier
    _HAS_XGB = True
except ImportError:
    _HAS_XGB = False
    warnings.warn('xgboost not installed — XGBoost cells will be skipped.')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.options.display.float_format = '{:.4f}'.format
np.random.seed(42)
%matplotlib inline

### 1b. Load data and filter to primary set

We load `dataset_clean_temporal.csv` — this is the cleaned dataset with the 7 extra calendar features already appended. We then restrict to the **primary set** (`is_extreme_event == 0`), which excludes the COVID distortion window (Feb-May 2020). This matches the convention used in every other model notebook in the project. The print statements confirm row counts, ticker coverage, date range, and overall `GapUp` rate.

In [ ]:
DATASET_PATH = '../structured_csv_data_files/fetched_data/dataset_clean_temporal.csv'
PLOTS_PATH   = '../structured_csv_data_files/fetched_data/plots/'
TARGET       = 'GapUp'

df_all = pd.read_csv(DATASET_PATH)
df_all['Date'] = pd.to_datetime(df_all['Date'], utc=True).dt.tz_convert(None)

primary = df_all[df_all['is_extreme_event'] == 0].copy()
primary = primary.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f'Total rows (all):  {len(df_all):,}')
print(f'Primary rows:      {len(primary):,}  (is_extreme_event == 0)')
print(f'Tickers:           {primary["Ticker"].nunique()}')
print(f'Date range:        {primary["Date"].min().date()}  ->  {primary["Date"].max().date()}')
print(f'GapUp rate:        {primary[TARGET].mean():.3f}')
print(f'\nAll columns ({len(df_all.columns)}):')
print(list(df_all.columns))

## 2. EDA Plots

Pre-generated temporal EDA figures. Each plot was produced from the full dataset (all tickers, 2016-2024) before COVID filtering.

### 2a. Display all 9 EDA figures

We scan the `plots/` directory for PNGs and render each one inline. The 9 figures cover:
- **Fig 1** — annual return distribution (how spread out weekly returns are by year)
- **Fig 2** — seasonality heatmap (month × year mean gap)
- **Fig 3** — quarterly return profile (Q1-Q4 average gap)
- **Fig 4** — volatility regime over time (rolling ATR or similar)
- **Fig 5** — week-of-year gap rate profile (which ISO weeks see more GapUps)
- **Fig 6** — ticker × year GapUp heatmap (cross-sectional seasonality)
- **Fig 7** — RSI temporal pattern (how RSI regime correlates with subsequent gap)
- **Fig 8** — gap extreme events (COVID spike and other outliers)
- **Fig 9** — rolling indicator correlation (how feature-target correlations shift over time)

These plots motivate whether calendar-based features are worth adding to the model.

In [ ]:
plot_files = sorted([f for f in os.listdir(PLOTS_PATH) if f.endswith('.png')])

plot_titles = {
    'fig1_annual_return_distribution.png':   'Fig 1 - Annual Return Distribution',
    'fig2_seasonality_heatmap.png':           'Fig 2 - Seasonality Heatmap (Month x Year)',
    'fig3_quarterly_return_profile.png':      'Fig 3 - Quarterly Return Profile',
    'fig4_volatility_regime.png':             'Fig 4 - Volatility Regime Over Time',
    'fig5_week_of_year_profile.png':          'Fig 5 - Week-of-Year Gap Rate Profile',
    'fig6_ticker_annual_heatmap.png':         'Fig 6 - Ticker x Year GapUp Heatmap',
    'fig7_rsi_temporal.png':                  'Fig 7 - RSI Temporal Pattern',
    'fig8_gap_extreme_events.png':            'Fig 8 - Gap Extreme Events',
    'fig9_rolling_indicator_correlation.png': 'Fig 9 - Rolling Indicator Correlation',
}

for fname in plot_files:
    title = plot_titles.get(fname, fname)
    img = mpimg.imread(PLOTS_PATH + fname)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()

## 3. Temporal Feature Stats

The `dataset_clean_temporal.csv` adds seven calendar-derived features:

| Feature | Description |
|---------|-------------|
| `Month_sin` / `Month_cos` | Cyclic encoding of month (period = 12) |
| `WeekOfYear_sin` / `WeekOfYear_cos` | Cyclic encoding of ISO week (period = 52) |
| `Quarter_sin` / `Quarter_cos` | Cyclic encoding of quarter (period = 4) |
| `DaysSinceStart` | Days since 2016-04-18 — a linear trend term |

Cyclic encodings preserve the circular topology (week 52 is adjacent to week 1) that a raw integer would break.

### 3a. Correlation of temporal features with GapUp

We compute the **point-biserial correlation** between each temporal feature and the binary `GapUp` target. This is the appropriate correlation measure when one variable is continuous and the other is binary — it is equivalent to a Pearson correlation in this case.

A significant correlation (low p-value, non-negligible |r|) tells us the calendar feature has a linear relationship with the gap direction. Even small |r| values can be useful in combination with other features — the walk-forward evaluation will determine whether they improve out-of-sample AUC.

In [ ]:
TEMPORAL_FEATURES = [
    'Month_sin', 'Month_cos',
    'WeekOfYear_sin', 'WeekOfYear_cos',
    'Quarter_sin', 'Quarter_cos',
    'DaysSinceStart',
]

print('Missing values in temporal features (primary set):')
print(primary[TEMPORAL_FEATURES].isna().sum())
print()

rows = []
for f in TEMPORAL_FEATURES:
    r, p = pointbiserialr(primary[TARGET], primary[f])
    rows.append({'Feature': f, 'Corr with GapUp': round(r, 4), 'p-value': round(p, 4)})
corr_df = pd.DataFrame(rows).set_index('Feature').sort_values('Corr with GapUp', key=abs, ascending=False)
print('Point-biserial correlation with GapUp:')
display(corr_df)

### 3b. GapUp rate by Month, Quarter, and Year

We group the primary set by each calendar dimension and compute the mean `GapUp` rate. The red dashed line shows the overall mean (~0.516). Bars above that line are calendar periods where gap-ups are more frequent than average — these are the periods our cyclic features are trying to encode.

Key things to look for:
- **Month plot**: Do certain months (e.g. January, earnings months) show consistently higher gap rates?
- **Quarter plot**: Is Q1 (post-year-end repositioning) or Q4 (tax-loss harvesting) different?
- **Year plot**: Is there a structural drift — has the gap rate changed post-2020?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

month_gap = primary.groupby('Month')[TARGET].mean().reset_index()
axes[0].bar(month_gap['Month'], month_gap[TARGET], color='steelblue')
axes[0].axhline(primary[TARGET].mean(), color='red', linestyle='--', label='Overall mean')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('GapUp rate')
axes[0].set_title('GapUp Rate by Month'); axes[0].legend()
axes[0].set_xticks(range(1, 13))

q_gap = primary.groupby('Quarter')[TARGET].mean().reset_index()
axes[1].bar(q_gap['Quarter'], q_gap[TARGET], color='darkorange')
axes[1].axhline(primary[TARGET].mean(), color='red', linestyle='--', label='Overall mean')
axes[1].set_xlabel('Quarter'); axes[1].set_ylabel('GapUp rate')
axes[1].set_title('GapUp Rate by Quarter'); axes[1].legend()

yr_gap = primary.groupby('Year')[TARGET].mean().reset_index()
axes[2].bar(yr_gap['Year'].astype(str), yr_gap[TARGET], color='seagreen')
axes[2].axhline(primary[TARGET].mean(), color='red', linestyle='--', label='Overall mean')
axes[2].set_xlabel('Year'); axes[2].set_ylabel('GapUp rate')
axes[2].set_title('GapUp Rate by Year'); axes[2].legend()
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Seasonal GapUp rates (primary set, COVID excluded)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3c. Rolling GapUp rate over time — checking for base-rate drift

We compute a rolling mean `GapUp` rate with a window of 52 weeks × 25 tickers (1,300 observations) — approximately one full calendar year of data across all stocks.

This answers a critical question: **does the base rate of GapUp change over our 9-year sample?** If it drifts substantially, a model trained on early years may be calibrated to the wrong base rate when predicting later years. This motivates the walk-forward expanding-window approach rather than a single train/test split.

In [ ]:
trend = primary.sort_values('DaysSinceStart').copy()
trend['GapUp_rolling'] = trend[TARGET].rolling(window=52 * 25, min_periods=200).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(trend['Date'], trend['GapUp_rolling'], color='steelblue', linewidth=1.5)
ax.axhline(primary[TARGET].mean(), color='red', linestyle='--', label='Overall mean')
ax.set_xlabel('Date'); ax.set_ylabel('Rolling GapUp rate')
ax.set_title('Rolling GapUp Rate Over Time (window = 52 weeks x 25 tickers)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Sets

Three feature sets evaluated:

| Set | Features | Rationale |
|-----|----------|----------|
| **Baseline** | 15 momentum/trend/volatility/volume/fundamental | Project baseline (post-VIF pruning) |
| **Temporal-only** | 7 cyclic + linear calendar features | Pure calendar signal |
| **Combined** | Baseline 15 + Temporal 7 = 22 features | Does temporal lift baseline? |

### 4a. Define the three feature sets

The **baseline 15** features are the same post-VIF-pruning set used by `logistic_regression.ipynb` and `xgboost_model.ipynb` — grouped into momentum (MACD, ROC, StochPercK), trend (CloseVEma50, CloseVSma20, ADX), volatility (BollingerBandWidth, ATR, FiveDStdDev), volume (OBV, MFI, VolumeRatio), and fundamental (NetMargin, RoA, RevGrowthQoQ).

The **temporal 7** features are all calendar-derived — no price or volume information — so they are guaranteed leak-free. The assertion at the end confirms there are no NaN values before we proceed to modelling.

In [ ]:
BASELINE_FEATURES = [
    'MACD', 'ROC', 'StochPercK',
    'CloseVEma50', 'CloseVSma20', 'ADX',
    'BollingerBandWidth', 'ATR', 'FiveDStdDev',
    'OBV', 'MFI', 'VolumeRatio',
    'NetMargin', 'RoA', 'RevGrowthQoQ',
]

TEMPORAL_FEATURES = [
    'Month_sin', 'Month_cos',
    'WeekOfYear_sin', 'WeekOfYear_cos',
    'Quarter_sin', 'Quarter_cos',
    'DaysSinceStart',
]

COMBINED_FEATURES = BASELINE_FEATURES + TEMPORAL_FEATURES

assert primary[BASELINE_FEATURES + TEMPORAL_FEATURES].isna().sum().sum() == 0, 'NaN found in features'
print(f'Baseline features:  {len(BASELINE_FEATURES)}')
print(f'Temporal features:  {len(TEMPORAL_FEATURES)}')
print(f'Combined features:  {len(COMBINED_FEATURES)}')

## 5. Walk-forward Evaluation

Fold structure identical to `logistic_regression.ipynb` / `xgboost_model.ipynb`:

| Fold | Train years | Test year | Note |
|------|-------------|----------|------|
| 1 | 2016-2018 | 2019 | |
| 2 | 2016-2019 | 2020 | COVID fold |
| 3 | 2016-2020 | 2021 | |
| 4 | 2016-2021 | 2022 | |
| 5 | 2016-2022 | 2023 | |
| 6 | 2016-2023 | 2024 | |

LogReg uses `StandardScaler + L2` pipeline. XGBoost is scale-invariant.

### 5a. Fold structure and model helpers

We define the 6-fold expanding-window split and a generic `run_walk_forward` function that accepts any feature list and model factory. This keeps the evaluation logic identical across all three feature sets and both classifiers.

**LogReg pipeline**: `StandardScaler` is required here because `DaysSinceStart` is in the thousands while the cyclic features are in [-1, 1] — without scaling, L2 regularisation penalises the large-scale feature disproportionately.

**XGBoost**: scale-invariant by construction (tree splits use rank order, not magnitude), so no scaler is needed. Conservative settings (`max_depth=4`, `learning_rate=0.05`, `n_estimators=200`) match the project's other XGBoost notebooks.

In [ ]:
FOLDS = [
    {'train_years': list(range(2016, 2019)), 'test_year': 2019},
    {'train_years': list(range(2016, 2020)), 'test_year': 2020},
    {'train_years': list(range(2016, 2021)), 'test_year': 2021},
    {'train_years': list(range(2016, 2022)), 'test_year': 2022},
    {'train_years': list(range(2016, 2023)), 'test_year': 2023},
    {'train_years': list(range(2016, 2024)), 'test_year': 2024},
]

anchor_year = primary['Year'].to_numpy()
anchor_y    = primary[TARGET].to_numpy().astype(int)


def run_walk_forward(features, make_model, label):
    X_all = primary[features].to_numpy(dtype=float)
    results = []
    for fold in FOLDS:
        tr_mask = np.isin(anchor_year, fold['train_years'])
        te_mask = (anchor_year == fold['test_year'])
        X_tr, X_te = X_all[tr_mask], X_all[te_mask]
        y_tr, y_te = anchor_y[tr_mask], anchor_y[te_mask]
        if len(y_te) == 0 or len(np.unique(y_te)) < 2:
            continue
        clf = make_model()
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1] if hasattr(clf, 'predict_proba') else clf.decision_function(X_te)
        pred  = (proba >= 0.5).astype(int)
        results.append({
            'Model':      label,
            'Test year':  fold['test_year'],
            'Train rows': int(tr_mask.sum()),
            'Test rows':  int(te_mask.sum()),
            'AUC':        round(roc_auc_score(y_te, proba), 4),
            'Accuracy':   round(accuracy_score(y_te, pred), 4),
            'COVID fold': fold['test_year'] == 2020,
        })
    return pd.DataFrame(results)


def make_logreg():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(penalty='l2', C=1.0, solver='liblinear',
                                      max_iter=1000, random_state=42)),
    ])


def make_xgb():
    return XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric='auc', tree_method='hist', random_state=42, n_jobs=-1,
    )


print('Walk-forward helper defined. Running evaluations...')

### 5b. Logistic Regression — run all three feature sets

We run the same walk-forward loop three times with the three feature sets. The output pivot table shows AUC per fold side-by-side, plus two delta columns:
- **Delta Comb-Base**: how much Combined beats Baseline each year
- **Delta Temp-Base**: how much Temporal-only beats/loses to Baseline each year

The mean AUC across the 5 non-COVID folds is the headline comparison metric — it tells us whether temporal features systematically add value across different market regimes.

In [ ]:
print('=== Logistic Regression ===')
lr_base     = run_walk_forward(BASELINE_FEATURES, make_logreg, 'LR-Baseline')
lr_temporal = run_walk_forward(TEMPORAL_FEATURES, make_logreg, 'LR-Temporal')
lr_combined = run_walk_forward(COMBINED_FEATURES, make_logreg, 'LR-Combined')

nc = ~lr_base['COVID fold'].values

lr_pivot = pd.DataFrame({
    'Test year':        lr_base['Test year'].values,
    'LR-Baseline AUC': lr_base['AUC'].values,
    'LR-Temporal AUC': lr_temporal['AUC'].values,
    'LR-Combined AUC': lr_combined['AUC'].values,
    'COVID fold':       lr_base['COVID fold'].values,
})
lr_pivot['Delta Comb-Base'] = (lr_pivot['LR-Combined AUC'] - lr_pivot['LR-Baseline AUC']).round(4)
lr_pivot['Delta Temp-Base'] = (lr_pivot['LR-Temporal AUC'] - lr_pivot['LR-Baseline AUC']).round(4)

print('\nPer-fold AUC:')
display(lr_pivot)

print(f'\nMean AUC (non-COVID folds):')
print(f'  LR-Baseline: {lr_base["AUC"].values[nc].mean():.4f}')
print(f'  LR-Temporal: {lr_temporal["AUC"].values[nc].mean():.4f}')
print(f'  LR-Combined: {lr_combined["AUC"].values[nc].mean():.4f}')

### 5c. XGBoost — run all three feature sets

Same walk-forward evaluation repeated with XGBoost. Because XGBoost can model **non-linear interactions** between temporal and technical features (e.g. high RSI in January vs high RSI in October may mean different things), we expect the Combined set to show a larger lift here than in LogReg — if temporal features carry non-linear signal.

The cell is gated behind `_HAS_XGB` so it skips cleanly if xgboost is not installed.

In [ ]:
if _HAS_XGB:
    print('=== XGBoost ===')
    xgb_base     = run_walk_forward(BASELINE_FEATURES, make_xgb, 'XGB-Baseline')
    xgb_temporal = run_walk_forward(TEMPORAL_FEATURES, make_xgb, 'XGB-Temporal')
    xgb_combined = run_walk_forward(COMBINED_FEATURES, make_xgb, 'XGB-Combined')

    nc_x = ~xgb_base['COVID fold'].values

    xgb_pivot = pd.DataFrame({
        'Test year':         xgb_base['Test year'].values,
        'XGB-Baseline AUC': xgb_base['AUC'].values,
        'XGB-Temporal AUC': xgb_temporal['AUC'].values,
        'XGB-Combined AUC': xgb_combined['AUC'].values,
        'COVID fold':        xgb_base['COVID fold'].values,
    })
    xgb_pivot['Delta Comb-Base'] = (xgb_pivot['XGB-Combined AUC'] - xgb_pivot['XGB-Baseline AUC']).round(4)
    xgb_pivot['Delta Temp-Base'] = (xgb_pivot['XGB-Temporal AUC'] - xgb_pivot['XGB-Baseline AUC']).round(4)

    print('\nPer-fold AUC:')
    display(xgb_pivot)

    print(f'\nMean AUC (non-COVID folds):')
    print(f'  XGB-Baseline: {xgb_base["AUC"].values[nc_x].mean():.4f}')
    print(f'  XGB-Temporal: {xgb_temporal["AUC"].values[nc_x].mean():.4f}')
    print(f'  XGB-Combined: {xgb_combined["AUC"].values[nc_x].mean():.4f}')
else:
    print('xgboost not installed — skipping.')
    xgb_base = xgb_temporal = xgb_combined = None

## 6. Results Comparison

### 6a. LogReg — per-fold AUC bar chart

Three grouped bars per fold (Baseline in blue, Temporal-only in orange, Combined in green). The dotted horizontal lines show the mean AUC over non-COVID folds for Baseline and Combined — the gap between these two lines is the headline lift number. The shaded column at 2020 marks the COVID fold, which is shown for reference but excluded from mean calculations.

A consistent pattern where the green bar sits above the blue bar across most folds would confirm that temporal features reliably add value.

In [ ]:
years = lr_base['Test year'].tolist()
x = np.arange(len(years))
width = 0.26

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, lr_base['AUC'],     width, label='LR-Baseline', color='steelblue')
ax.bar(x,         lr_temporal['AUC'], width, label='LR-Temporal', color='#ff7f0e')
ax.bar(x + width, lr_combined['AUC'], width, label='LR-Combined', color='seagreen')

ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Chance (0.5)')
nc_m = ~lr_base['COVID fold'].values
nc_base = lr_base['AUC'].values[nc_m].mean()
nc_comb = lr_combined['AUC'].values[nc_m].mean()
ax.axhline(nc_base, color='steelblue', linestyle=':', linewidth=1.2,
           label=f'LR-Baseline mean={nc_base:.3f}')
ax.axhline(nc_comb, color='seagreen', linestyle=':', linewidth=1.2,
           label=f'LR-Combined mean={nc_comb:.3f}')

covid_pos = years.index(2020)
ax.axvspan(covid_pos - 0.5, covid_pos + 0.5, alpha=0.08, color='red', label='COVID fold')

ax.set_xticks(x); ax.set_xticklabels(years)
ax.set_xlabel('Test year'); ax.set_ylabel('AUC')
ax.set_title('Logistic Regression - per-fold AUC by feature set')
ax.set_ylim(0.40, 0.75)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

### 6b. XGBoost — per-fold AUC bar chart

Same layout as the LogReg chart. Comparing the two charts side-by-side tells us whether temporal features help more with a linear model (LogReg) or a tree-based one (XGBoost). Trees can exploit non-linear interactions between calendar position and technical indicators, so we might see a bigger Combined lift here.

In [ ]:
if _HAS_XGB and xgb_base is not None:
    years_x = xgb_base['Test year'].tolist()
    x = np.arange(len(years_x))

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - width, xgb_base['AUC'],     width, label='XGB-Baseline', color='steelblue')
    ax.bar(x,         xgb_temporal['AUC'], width, label='XGB-Temporal', color='#ff7f0e')
    ax.bar(x + width, xgb_combined['AUC'], width, label='XGB-Combined', color='seagreen')

    ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Chance (0.5)')
    nc_xm = ~xgb_base['COVID fold'].values
    nc_xb = xgb_base['AUC'].values[nc_xm].mean()
    nc_xc = xgb_combined['AUC'].values[nc_xm].mean()
    ax.axhline(nc_xb, color='steelblue', linestyle=':', linewidth=1.2,
               label=f'XGB-Baseline mean={nc_xb:.3f}')
    ax.axhline(nc_xc, color='seagreen', linestyle=':', linewidth=1.2,
               label=f'XGB-Combined mean={nc_xc:.3f}')

    covid_pos = years_x.index(2020)
    ax.axvspan(covid_pos - 0.5, covid_pos + 0.5, alpha=0.08, color='red', label='COVID fold')

    ax.set_xticks(x); ax.set_xticklabels(years_x)
    ax.set_xlabel('Test year'); ax.set_ylabel('AUC')
    ax.set_title('XGBoost - per-fold AUC by feature set')
    ax.set_ylim(0.40, 0.75)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()

### 6c. Lift delta chart — Combined minus Baseline per fold

This chart isolates the **incremental contribution** of the temporal features by plotting `AUC(Combined) - AUC(Baseline)` for each fold. Green bars = temporal features helped that year; red bars = they hurt (or overfitted).

The delta labels printed above each bar make it easy to quantify the lift. We want to see mostly green bars and a positive average — that would justify keeping the temporal features in the final model.

In [ ]:
n_plots = 2 if (_HAS_XGB and xgb_base is not None) else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 4), sharey=True)
if n_plots == 1:
    axes = [axes]

delta_lr = lr_combined['AUC'].values - lr_base['AUC'].values
colors_lr = ['seagreen' if d > 0 else 'tomato' for d in delta_lr]
axes[0].bar(lr_base['Test year'].astype(str), delta_lr, color=colors_lr)
axes[0].axhline(0, color='black', linewidth=0.8)
for i, d in enumerate(delta_lr):
    axes[0].text(i, d + (0.001 if d >= 0 else -0.003), f'{d:+.3f}', ha='center', fontsize=8)
axes[0].set_title('LogReg: ΔAUC (Combined - Baseline) per fold')
axes[0].set_xlabel('Test year'); axes[0].set_ylabel('ΔAUC')

if n_plots > 1:
    delta_xgb = xgb_combined['AUC'].values - xgb_base['AUC'].values
    colors_xgb = ['seagreen' if d > 0 else 'tomato' for d in delta_xgb]
    axes[1].bar(xgb_base['Test year'].astype(str), delta_xgb, color=colors_xgb)
    axes[1].axhline(0, color='black', linewidth=0.8)
    for i, d in enumerate(delta_xgb):
        axes[1].text(i, d + (0.001 if d >= 0 else -0.003), f'{d:+.3f}', ha='center', fontsize=8)
    axes[1].set_title('XGBoost: ΔAUC (Combined - Baseline) per fold')
    axes[1].set_xlabel('Test year')

plt.suptitle('Temporal feature lift over baseline (green = positive lift)', fontsize=12)
plt.tight_layout()
plt.show()

### 6d. Final summary table

Mean AUC and mean accuracy across the 5 non-COVID folds for every model × feature-set combination. This is the single table to cite when reporting results. The COVID fold (2020) is excluded because the distributional shift during that period is attributable to the macro event, not to the feature set choice.

In [ ]:
rows = []
nc_m = ~lr_base['COVID fold'].values
for df, label in [(lr_base, 'LR-Baseline'), (lr_temporal, 'LR-Temporal'), (lr_combined, 'LR-Combined')]:
    rows.append({
        'Model':         label,
        'Mean AUC':      round(df['AUC'].values[nc_m].mean(), 4),
        'Mean Accuracy': round(df['Accuracy'].values[nc_m].mean(), 4),
        'n folds (excl COVID)': int(nc_m.sum()),
    })

if _HAS_XGB and xgb_base is not None:
    nc_xm = ~xgb_base['COVID fold'].values
    for df, label in [(xgb_base, 'XGB-Baseline'), (xgb_temporal, 'XGB-Temporal'), (xgb_combined, 'XGB-Combined')]:
        rows.append({
            'Model':         label,
            'Mean AUC':      round(df['AUC'].values[nc_xm].mean(), 4),
            'Mean Accuracy': round(df['Accuracy'].values[nc_xm].mean(), 4),
            'n folds (excl COVID)': int(nc_xm.sum()),
        })

summary = pd.DataFrame(rows)
print('Summary - Mean AUC across non-COVID folds (2019, 2021, 2022, 2023, 2024):')
display(summary)

## 7. Summary

### Key questions answered

| Question | Where to look |
|----------|---------------|
| Do temporal features carry gap signal on their own? | `LR-Temporal` / `XGB-Temporal` mean AUC vs 0.50 |
| Do temporal features lift the baseline? | `Delta Comb-Base` column in pivot tables |
| Is the lift consistent across folds? | Per-fold delta bar chart |

### Interpretation guidance

- **Temporal-only AUC > 0.50** means the market's gap behaviour is partly predictable from the calendar alone (earnings seasons, year-end effects, quarter-turn repositioning).
- **Combined > Baseline** means temporal features carry *additional* signal beyond what technical/fundamental indicators already capture. The mean ΔAUC across non-COVID folds is the number to report.
- **Inconsistent fold-by-fold delta** suggests the calendar effect is regime-dependent — helped in some years but not others.
- **COVID fold (2020)** is reported but excluded from mean AUC comparisons.

### Extension points

- **Earnings-week indicator** — a binary `is_earnings_week` flag would be a stronger temporal signal than cyclic month/week encodings.
- **Interaction features** — `Month_sin x RSI`, `Quarter_cos x ADX` — to capture conditional seasonality of technical indicators.
- **Per-ticker seasonal GapUp rates** — fit separate logistic intercepts per ticker-month to capture idiosyncratic earnings calendars.
- **Compare against KarmaLego/KLS features** — see `temporal_analysis.ipynb` for the multi-week symbolic pattern approach.